**MONTE CARLO AGGREGATION FOR WEIGHTING**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from datetime import datetime

BANDITS CLASS

In [2]:
# class for bandits K armed
class GaussianBandit:
    def __init__(self, means, sigma):
        #f are estiamtes of real mean
        self.f = np.ones(len(means))*0.5
        self.means = np.array(means)
        self.eps = sigma
        self.K = len(means)
        self.pulls = np.zeros(len(means))
        self.T = 0
        self.rad = np.ones(len(means))
        self.UCB = np.ones(len(means))*0.5 + np.ones(len(means))
        self.steps = np.zeros(self.K)

    def pull(self, arm):
        noise = np.random.normal(0, self.eps)
        return self.means[arm] + noise

    def update(self, reward, k):
        self.T += 1
        self.pulls[k] += 1
        self.f[k] += (reward - self.f[k]) / self.pulls[k]
        self.rad[k] = np.sqrt(2 * np.log(self.T) / self.pulls[k])
        self.UCB[k] = self.f[k] + self.rad[k]
    
    #aux function for elimination arm
    def update_elimination(self, reward, k, h, epsh):
        self.T += 1
        self.pulls[k] += 1
        self.f[k] += (reward - self.f[k]) / self.pulls[k]
        self.rad[k] = np.sqrt(2 * np.log(h+1) / self.pulls[k])
        self.UCB[k] = self.f[k] + self.rad[k]
        if self.pulls[k] >= 1/epsh**2:
            self.steps[k]=h+1

    #deletes data from the bandits
    def nullify_bandit(self):
        self.f = np.ones(len(self.means))*0.5
        self.pulls = np.zeros(len(self.means))
        self.T = 0
        self.rad = np.ones(len(self.means))
        self.UCB = np.ones(len(self.means))*0.5 + np.ones(len(self.means))
        self.steps = np.zeros(len(self.means))

DATA GENERATION FOR PERFECT LUMPING

In [3]:
def generate_lumpable_means_permuted( N, K, L, min_gap=0.2, seed=None):

    delta = 0
    if seed is not None:
        np.random.seed(seed)

    assert delta < min_gap / 2, "delta too large"

    # ---- Base decreasing profile (shared by all clusters)
    gaps = min_gap + np.random.uniform(0, min_gap, size=K-1)
    base = 1.0 - np.cumsum(np.concatenate([[0], gaps]))

    # Assign contexts to clusters
    clusters = np.repeat(np.arange(L), N // L)
    clusters = np.pad(clusters, (0, N - len(clusters)), mode="edge")

    means = np.zeros((N, K))
    perms = []

    # ---- First cluster permutation
    perms.append(np.random.permutation(K))

    # ---- Remaining clusters: enforce >= K/2 disagreement
    for l in range(1, L):
        while True:
            p = np.random.permutation(K)
            hamming = np.sum(p != perms[0])
            if hamming >= K :
                perms.append(p)
                break

    # ---- Build clusters
    for l in range(L):
        perm = perms[l]
        cluster_vector = base[perm]

        # optimal arm index in observed coordinates
        a_star = np.argmax(cluster_vector)

        idxs = np.where(clusters == l)[0]

        for i in idxs:
            noise = np.zeros(K)
            subopt = np.arange(K) != a_star
            noise[subopt] = np.random.uniform(-delta/2, delta/2, size=np.sum(subopt))

            means[i] = cluster_vector + noise

    means = np.clip(means, 0.0, 1.0)

    return means, clusters

FUNCTIONS FOR WEIGHTS

In [4]:
#function aux for distances
def Delta_fun(f1, f2):
    f1 = np.asarray(f1)
    f2 = np.asarray(f2)
    return np.sum((f1 - f2) ** 2)

# function to compute actual Delta
def Delta_bandit(bandits, S, Delta, eps, h):
    N = len(bandits)
    c = np.matmul(S, S.T) #how much overlap 

    DELTA_NEUTRAL = 0.0  # if the exploration can't tell
    #loop on ontexts
    for i in range(N):
        for j in range(N):
            # active overlap
            overlapping = np.where((S[i] == 1) & (S[j] == 1))[0]
            # no overlap at all: then no same cluster
            if len(overlapping) == 0:
                Delta[i, j] = np.inf
                continue
            # sufficiently explored overlap see where
            valid = [
                k for k in overlapping
                if bandits[i].steps[k] >= h+1 and bandits[j].steps[k] >= h+1
            ]

            #overlap exists, but insufficient exploration
            if len(valid) == 0:
                Delta[i, j] = DELTA_NEUTRAL
                continue

            # overlap + sufficient exploration, then select the vector of feedbacks
            fi = [bandits[i].f[k] for k in valid]
            fj = [bandits[j].f[k] for k in valid]
            
            dist = Delta_fun(fi, fj)

            if c[i, j] > 0: #weighted distances
                Delta[i, j] = dist / (c[i, j] *4* eps**2)
            else: #if this was inf same
                Delta[i, j] = dist

    return Delta


In [5]:
def update_weights(W, Delta):#here is the version not regularized
    eta = 0.7
    #alpha = min(0.1, eps)
    # multiplicative update 
    W_new = W * np.exp(-eta * Delta)

    # force exact zeros where no overlap
    W_new[np.isinf(Delta)] = 0.0

    # row-normalize
    row_sums = W_new.sum(axis=1, keepdims=True)
    # numerical safety
    row_sums[row_sums == 0] = 1.0
    W_new = W_new / row_sums
    print("weights updated")
    return W_new

In [6]:
#THIS VERSION WORKING WITH SAME OPTIMA
def arm_elimination(bandits, W, S):
    N = len(bandits)
    K = bandits[0].K
    print("eliminate weights arms")

    UCBs = np.array([bandits[b].UCB for b in range(N)])
    rads = np.array([bandits[b].rad for b in range(N)])
    
    LCBs = UCBs - 2*rads
    maxLCBs = np.max(LCBs, axis=1) 
    maxUCBs = np.max(UCBs, axis=1) 

    for b in range(N):
        for k in range(K):
            if S[b][k] == 1 and UCBs[b][k] < np.matmul(W[b], maxLCBs): 
                S[b][k] = 0
            if np.sum(S[b]) == 0:
                best_arm = np.argmax(UCBs[b])
                S[b][best_arm] = 1
    #print("n:", bandits[0].pulls)
    #print("rad:", bandits[0].rad)
    #print("UCB:", bandits[0].UCB)
    #print("LCB:", bandits[0].UCB - bandits[0].rad)
    #print("maxLCB:", np.max(bandits[0].UCB - 2*bandits[0].rad))
    #print("W:", W)
    return S

In [7]:
def explore_bandits(bandits, T, S, eps, optimal_arms, h):
    print("exploration phase for weights")
    N = len(bandits)
    regrets = np.zeros(T)
    chosen_index = [None] * N
    # random active arm per context
    for b in range(N):
        active = np.where(S[b] == 1)[0]
        if len(active) == 0:
            continue
        assert S.shape[1] == bandits[b].K, (
            f"S has {S.shape[1]} arms, bandit has {bandits[b].K}"
        )
        chosen_index[b] = np.random.choice(active)
    # when should I change
    quota = int(1 / eps**2)
    #loop exploration
    for t in range(T):
        # select context
        b = np.random.randint(N)
        xt = chosen_index[b]
        if xt is None:
            continue
        # pull arm
        rewt = bandits[b].pull(xt)
        bandits[b].update_elimination(rewt, xt, h, eps)
        regrets[t] = optimal_arms[b] - rewt

        # check if this arm is explored enough
        if bandits[b].pulls[xt] >= quota:
            active = np.where(S[b] == 1)[0]
            # arms still under-explored
            under_explored = [
                k for k in active
                if bandits[b].pulls[k] < quota
            ]
            if len(under_explored) > 0:
                chosen_index[b] = np.random.choice(under_explored)
            else:
                chosen_index[b] = np.random.choice(active)

    return regrets


FUNCTIONS FOR LUMPABLE

In [8]:
def sufficiently_explored_by_context(bandits, eps):
    Data = {}
    for b, bandit in enumerate(bandits):
        arms = np.where(bandit.pulls >= 1/ eps**2)[0]
        if len(arms) > 0:
            Data[b] = arms.tolist()
    return Data

In [9]:
def explore_lumpable(bandits, Lh, S, eps, optimal_arms, h):
    N = len(bandits)
    regrets = []
    print("exploring times:", Lh)
    chosen_index = [None] * N
    quota = int(np.ceil(1 / eps**2))

    for b in range(N):
        active = np.where(S[b] == 1)[0]
        if len(active) > 0:
            chosen_index[b] = np.random.choice(active)

    for _ in range(Lh):
        b = np.random.randint(N)
        xt = chosen_index[b]
        if xt is None:
            continue

        reward = bandits[b].pull(xt)
        bandits[b].update_elimination(reward, xt, h, eps)
        regrets.append(optimal_arms[b] - reward)

        if bandits[b].pulls[xt] >= quota: #update choice of arm
            active = np.where(S[b] == 1)[0]
            under = [k for k in active if bandits[b].pulls[k] < quota]
            chosen_index[b] = np.random.choice(under if under else active)

    return regrets

In [10]:
def arm_elimination_lumpable(bandits, Data, lump, eps_h):
    
    K = bandits[0].K
    lump = list(lump)
    print("eliminating arms")
    #compute per-arm lump values
    mu_lump = {}  

    for k in range(K): #extract the values for the arm
        vals = [bandits[c].f[k] for c in lump if (c in Data and k in Data[c])]
        if len(vals) > 0:
            mu_lump[k] = max(vals)

    if len(mu_lump) == 0:
        return set()

    mu_star = max(mu_lump.values())
    eliminated = {k for k, val in mu_lump.items() if mu_star - val > 2 * eps_h}

    return eliminated


In [11]:
def context_split_test(bandits, Data, lump, eps_tilde):

    K = bandits[0].K #number of arms
    lump = list(lump) #contexts
    print("checking for splits")
    for i in range(len(lump)): #for every context in lump
        for j in range(i + 1, len(lump)): #for every other context
            c1, c2 = lump[i], lump[j] 
            for k in range(K): #for every arm explored
                if (c1 in Data and k in Data[c1] and
                    c2 in Data and k in Data[c2]):
                    if abs(bandits[c1].f[k] - bandits[c2].f[k]) >= eps_tilde:
                        return c1, c2, k
    return None


In [12]:
#alg 5
def lump_up(bandits, eps_pr, delta_pr, k, S, lump, optimal_arms, h):

    N = len(bandits)
    expl_regrets = []
    # exploration length
    lg_tilde =2* np.log(N / delta_pr) #tolto 64
    L_pr = int(np.ceil(N * lg_tilde / eps_pr**2))

    #init lumps
    new_lumps = []
    print("exploring for lumping")
    #new data
    expl_regrets = explore_lumpable(bandits, L_pr, S, eps_pr, optimal_arms, h)
    
     # split threshold for lump up
    gap_thresh = np.sqrt(lg_tilde) * eps_pr

    # sort contexts by refined estimates for that arm
    ordered = sorted(lump, key=lambda c: bandits[c].f[k])

    # scan and split
    current_block = [ordered[0]]
    for i in range(1, len(ordered)):
        prev = ordered[i - 1] #before
        curr = ordered[i]     #after

        if abs(bandits[curr].f[k] - bandits[prev].f[k]) > gap_thresh:
            new_lumps.append(set(current_block))
            current_block = [curr]
            print("lumped up")
        else:
            current_block.append(curr)

    new_lumps.append(set(current_block))

    return new_lumps, expl_regrets


MAIN LOOP

In [13]:
#define main parameters
N = 32
L = 4
K = 64

#parameters for data generation
gap_scale=0.15
means = np.zeros([L,K]) 

##########################################################################
phases = 7
#steps per phase, precision
eps = [pow(2,-h/2) for h in range(phases)]
#number of exploratory phases
TW = [
    int(np.ceil(
        (N if epsh > gap_scale else L) * K * 64 * np.log(N) / epsh**2
    ))
    for epsh in eps
]

#number of experiments
draws = 10
regrets_W = np.zeros( [draws,np.sum(TW)])

 # things for lumpable bandits #############################################
regrets_L = np.zeros( [draws,np.sum(TW)])


# constants init
delta = [eps[h]**2 / (N * (L**3) * K) for h in range(phases)]
lg = [2* np.log((L * N * K) / delta[h]) for h in range(phases)] #tolto 64
eps_tilde = [np.sqrt(lg[h]) * eps[h] for h in range(phases)]
Lh = [int(np.ceil(L * (N + K) * lg[h] / (eps[h]**2))) for h in range(phases)]
nh = [np.log(1/eps[h]**2) for h in range(phases)]

In [14]:
for dr in range(draws): #loop on random instance draw
    regrets = []
    t0 = 0 #time count
    print("draw number:", dr+1 ,"/", draws)
    # init the matrices 
    W = np.ones([N,N])/N
    Delta = np.zeros([N,N])
    # active arms 0 o 1
    SW = np.ones([N,K])
    bandits = [None] * N
    optimal_arms = [None] * N
    SL = np.ones((N, K))
    lumpsL = [set(range(N))]
    
    #draw different instances ##################################################
    means, _ = generate_lumpable_means_permuted(N, K, L, gap_scale)
    
    #assigning the mean to lump exactly
    for con in range(N):
        bandits[con] = GaussianBandit(means=means[con] , sigma=0.05)
        optimal_arms[con] = np.max(means[con])
    #init bandits
    bandits_W = deepcopy(bandits)
    bandits_L = deepcopy(bandits)

    #MAIN LOOP ###############################################################
    for h in range(phases):
        # WEIGHTS ############################################################
        
        Th = TW[h]
        phase_regrets = explore_bandits( bandits_W, Th, SW, eps[h], optimal_arms, h)
        regrets_W[dr][t0 : t0 + Th] = phase_regrets
        t0 += Th
        
        #update variables
        Delta = Delta_bandit(bandits_W, SW, Delta, eps[h], h)
        W = update_weights(W, Delta) 
        
        #arm elimination
        SW = arm_elimination(bandits_W, W, SW)
        
        # LUMPABLE ################################################################
        # EXPLORATION
        phase_regrets = explore_lumpable(bandits_L, Lh[h], SL, eps[h], optimal_arms, h)
        regrets.extend(phase_regrets)
        Data = sufficiently_explored_by_context(bandits_L, eps[h])
    
        new_lumps = []

        for lump in lumpsL:
            split = context_split_test(bandits_L, Data, lump, eps_tilde[h])
    
            if split is None:
                new_lumps.append(lump)
                continue
    
            _, _, k = split
    
            # auxiliary S
            S_aux = SL.copy()
            S_aux[list(lump), :] = 0
            S_aux[list(lump), k] = 1
    
            lumps_refined, aux_regrets = lump_up(bandits_L, eps[h] / (4 * L), delta[h] / L, k, S_aux, lump, optimal_arms, h)
            regrets.extend(aux_regrets)
    
            # add refined lumps
            new_lumps.extend(lumps_refined)
        
        # replace partition
        lumpsL = new_lumps

        # ELIMINATION
        for lump in lumpsL:
            eliminated = arm_elimination_lumpable(bandits_L, Data, lump, eps[h])
            for c in lump:
                for k in eliminated:
                    SL[c, k] = 0
    ################################################################################
    #PRINTS
    print("Phase", h, "lumps:", lumpsL)
    #print(SL)
    T = min(len(regrets), regrets_L.shape[1])
    regrets_L[dr, :T] = regrets[:T]
    print("W:", W)
    # bandits reset approximates
    for n in range(N):
        bandits_W[n].nullify_bandit()
        bandits_L[n].nullify_bandit()


draw number: 1 / 10
exploration phase for weights


KeyboardInterrupt: 

SAVE FILE AND PRINT REGRET PLOT

In [ ]:
# timestamped filename
fname = f"bandit_results_N{N}_K{K}_L{L}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.npz"

np.savez(
    fname,
    regrets_W=regrets_W,
    regrets_L=regrets_L,
    eps=np.array(eps),
    TW=np.array(TW),
    Lh=np.array(Lh),
    N=N,
    K=K,
    L=L,
    gap_scale=gap_scale,
)

print("Saved to:", fname)

In [ ]:
cum_regrets = np.cumsum(regrets_W, axis=1)
mean_cum = np.mean(cum_regrets, axis=0)
std_cum = np.std(cum_regrets, axis=0)
t = np.arange(len(mean_cum))

cum_regretsL = np.cumsum(regrets_L, axis=1)
mean_cumL = np.mean(cum_regretsL, axis=0)
std_cumL = np.std(cum_regretsL, axis=0)
tL = np.arange(len(mean_cumL))
#rate dei regret
rate1 = []
rate2 = []
rate3 = []

for l in range(1,np.size(mean_cumL)):
    rate3.append(np.pow(l*L*(K+N)*np.log(N), 1/2))
    rate1.append(np.pow(l*N*L*np.log(N), 1/2))

#start figure new
plt.figure(figsize=(8, 5))

plt.plot(t, mean_cum, label="Mean cumulative regret weights")
plt.plot(tL, mean_cumL, label="Mean cumulative regret lumpable")
plt.fill_between(
    t,
    mean_cum - std_cum,
    mean_cum + std_cum,
    alpha=0.3,
    label="±1 std weights"
)
plt.fill_between(
    tL,
    mean_cumL - std_cumL,
    mean_cumL + std_cumL,
    alpha=0.3,
    label="±1 std lumpable"
)
plt.plot(rate3, label="rate of $\sqrt{L(N+K)T}$")
plt.xlabel("Time")
plt.ylabel("Cumulative regret")
plt.title("Cumulative regret over draws: weights and lumpable")
plt.legend()
plt.xlim(0, 300000)     
plt.ylim(0, 30000)       
plt.grid(True)
plt.tight_layout()

plt.show()
figname = f"bandit_regret_N{N}_K{K}_L{L}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jpg"
#plt.savefig(figname, dpi=200)
#plt.close()
